<a href="https://colab.research.google.com/github/Coolguy4123/Data-Visualization-Project-3/blob/main/CS4990_Vehicle_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Initialization

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("syedanwarafridi/vehicle-sales-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'vehicle-sales-data' dataset.
Path to dataset files: /kaggle/input/vehicle-sales-data


In [2]:
import pandas as pd
df = pd.read_csv(path + "/car_prices.csv")
df.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


# Data Preprocessing

In [3]:
print(df.info())

print(f"\n\nShape: {df.shape}")
print(f"Columns: {df.columns}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  object 
 2   model         548438 non-null  object 
 3   trim          548186 non-null  object 
 4   body          545642 non-null  object 
 5   transmission  493485 non-null  object 
 6   vin           558833 non-null  object 
 7   state         558837 non-null  object 
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  object 
 11  interior      558088 non-null  object 
 12  seller        558837 non-null  object 
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558825 non-null  object 
dtypes: float64(4), int64(1), object(11)
memory usage: 68.2+ MB
None


Shape: (558837, 16)
Column

In [4]:
print(f"Number of duplicates: {df.duplicated().sum()}")
print(f"\n\nNumber of missing values: {df.isnull().sum()}")

Number of duplicates: 0


Number of missing values: year                0
make            10301
model           10399
trim            10651
body            13195
transmission    65352
vin                 4
state               0
condition       11820
odometer           94
color             749
interior          749
seller              0
mmr                38
sellingprice       12
saledate           12
dtype: int64


In [5]:
# --- Data cleaning steps ---
# ---------------------------

# 1. Drop rows with missing values in important columns
df = df.dropna(subset=['sellingprice', 'mmr', 'odometer', 'make', 'body'])


# 2. Fill missing values in less critical columns
df['transmission'] = df['transmission'].fillna('unknown')
df['condition'] = df['condition'].fillna(df['condition'].median())
df['color'] = df['color'].fillna('unknown')
df['interior'] = df['interior'].fillna('unknown')


# 3. Drop columns that are not useful for visualization
df = df.drop(columns=['vin', 'seller', 'trim', 'model', 'saledate'])


# 4. Remove extreme outliers from key numeric columns
df = df[df['sellingprice'] < df['sellingprice'].quantile(0.99)]
df = df[df['odometer'] < df['odometer'].quantile(0.99)]
df = df.copy()


# 5. Reduce category complexity for cleaner visualizations
top_makes = df['make'].value_counts().nlargest(10).index
df['make'] = df['make'].where(df['make'].isin(top_makes), 'Other')


top_body = df['body'].value_counts().nlargest(8).index
df['body'] = df['body'].where(df['body'].isin(top_body), 'Other')


# 6. Columns for visualization
df = df[['year', 'make', 'body', 'transmission', 'condition',
         'odometer', 'color', 'interior', 'state', 'mmr',
         'sellingprice']]

print("\nNew Shape:", df.shape)
print("\nMissing values after preprocessing:\n", df.isnull().sum())


New Shape: (534662, 11)

Missing values after preprocessing:
 year            0
make            0
body            0
transmission    0
condition       0
odometer        0
color           0
interior        0
state           0
mmr             0
sellingprice    0
dtype: int64


In [6]:
# --- Encoding and Scaling ---
# ----------------------------
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
df = df.copy()

# --- 1. Encode categorical variables ---
categorical_cols = ['make', 'body', 'transmission', 'color', 'interior', 'state']

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# --- 2. Scale numerical variables ---
numerical_cols = ['year', 'condition', 'odometer', 'mmr', 'sellingprice']

scaler = MinMaxScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print(df.describe())
df.head()

                year           make           body   transmission  \
count  534662.000000  534662.000000  534662.000000  534662.000000   
mean        0.807481       5.948897       4.364090       0.260935   
std         0.152592       3.174192       1.684915       0.650555   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.720000       4.000000       3.000000       0.000000   
50%         0.880000       7.000000       5.000000       0.000000   
75%         0.920000       9.000000       5.000000       0.000000   
max         1.000000      10.000000       8.000000       2.000000   

           condition       odometer          color       interior  \
count  534662.000000  534662.000000  534662.000000  534662.000000   
mean        0.624099       0.292867       9.632112       4.010549   
std         0.274380       0.214237       6.943719       4.327312   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.479167       0.1261

,year,make,body,transmission,condition,odometer,color,interior,state,mmr,sellingprice
0,1.00,7,4,0,0.083333,0.074157,18,1,3,0.186179,0.478479
1,1.00,7,4,0,0.083333,0.041861,18,0,3,0.188907,0.478479
2,0.96,0,5,0,0.916667,0.005928,7,1,3,0.289839,0.667653
3,1.00,9,5,0,0.833333,0.063652,18,1,3,0.249830,0.617578
5,1.00,8,5,0,0.000000,0.024750,7,1,3,0.139350,0.242567


In [7]:
import pandas as pd
import plotly.express as px
import plotly.io as pio


# ensure these columns are numeric
numeric_cols = ["odometer", "mmr", "sellingprice"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# drop rows missing the required numeric values
df_viz = df.dropna(subset=numeric_cols)

# sample the data so the plot isn't insanely messy
df_sample = df_viz.sample(1000, random_state=42)

# convert make → numbers (parallel coords needs numeric colors)
df_sample["make_code"] = df_sample["make"].astype("category").cat.codes

# build the parallel-coordinates plot
fig = px.parallel_coordinates(
    df_sample,
    dimensions=["odometer", "mmr", "sellingprice"],
    color="make_code",
    labels={
        "odometer": "Odometer (miles)",
        "mmr": "MMR Value",
        "sellingprice": "Selling Price",
        "make_code": "Make (encoded)"
    },
    color_continuous_scale=px.colors.qualitative.Set2
)

fig.show()


Questions to consider:

*   Find a line that begins near the low end of the Odometer axis (close to 0). What approximate MMR Value and Selling Price does it connect to?  


*   Locate a line that starts high on the MMR Value axis (near 1). What approximate Selling Price does it end at on the Selling Price axis?

*   Do lines with higher Odometer values (toward the top of the Odometer axis) tend to connect to lower MMR Values? Describe the pattern you see.

*   Is there a visible relationship between MMR Value and Selling Price (for example, do higher MMR Values tend to lead to higher Selling Prices)? Explain what you observe.


